In [9]:
import sys
import numpy as np

print("Python:", sys.executable)
print("NumPy:", np.__version__)
import pandas as pd

print("Pandas:", pd.__version__)
import sys
print(sys.executable)
print("Apriori Algorithm ")

Python: C:\Users\asus\anaconda3\python.exe
NumPy: 2.5.3
Pandas: 3.0.5
C:\Users\asus\anaconda3\python.exe
Apriori Algorithm 


In [35]:
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("APRIORI------------------------------>")
# ============================================
# 1. TRANSACTION DATA
# ============================================

data = pd.DataFrame({
    "Transaction ID": [1, 2, 3, 4, 5],
    "Item 1": ["Milk", "Milk", "Eggs", "Milk", "Milk"],
    "Item 2": ["Bread", "Bread", "Bread", "", "Bread"],
    "Item 3": ["Butter", "", "", "Butter", "Eggs"]
})


# ============================================
# 2. REMOVE TRANSACTION ID
# ============================================

items = data.drop(columns=["Transaction ID"])


# ============================================
# 3. CONVERT DATA INTO TRANSACTIONS
# ============================================

transactions = items.values.tolist()

# Remove empty values
transactions = [
    [item for item in transaction if item != ""]
    for transaction in transactions
]

print("Transactions:")
print(transactions)


# ============================================
# 4. TRANSACTION ENCODER
# ============================================

te = TransactionEncoder()

te_ary = te.fit(transactions).transform(transactions)


# ============================================
# 5. CREATE BASKET DATAFRAME
# ============================================

basket_sets = pd.DataFrame(
    te_ary,
    columns=te.columns_
)

print("\nBasket:")
print(basket_sets)


# ============================================
# 6. APRIORI
# ============================================

frequent_itemsets = apriori(
    basket_sets,
    min_support=0.4,
    use_colnames=True
)

print("\nFrequent Itemsets:")
print(frequent_itemsets)


# ============================================
# 7. ASSOCIATION RULES
# ============================================

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

print("\nAssociation Rules:")

print(
    rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ]
)

APRIORI------------------------------>
Transactions:
[['Milk', 'Bread', 'Butter'], ['Milk', 'Bread'], ['Eggs', 'Bread'], ['Milk', 'Butter'], ['Milk', 'Bread', 'Eggs']]

Basket:
   Bread  Butter   Eggs   Milk
0   True    True  False   True
1   True   False  False   True
2   True   False   True  False
3  False    True  False   True
4   True   False   True   True

Frequent Itemsets:
   support                   itemsets
0      0.8         frozenset({Bread})
1      0.4        frozenset({Butter})
2      0.4          frozenset({Eggs})
3      0.8          frozenset({Milk})
4      0.4   frozenset({Eggs, Bread})
5      0.6   frozenset({Milk, Bread})
6      0.4  frozenset({Milk, Butter})

Association Rules:
           antecedents         consequents  support  confidence    lift
0    frozenset({Eggs})  frozenset({Bread})      0.4        1.00  1.2500
1    frozenset({Milk})  frozenset({Bread})      0.6        0.75  0.9375
2   frozenset({Bread})   frozenset({Milk})      0.6        0.75  0.9375
3  fr

In [37]:
import pandas as pd
from itertools import combinations
print("ECLAT : ------------------->")

# ============================================
# 1. SAMPLE TRANSACTION DATA
# ============================================

data = {
    "TransactionID": [
        1, 1, 1,
        2, 2, 2,
        3, 3,
        4, 4, 4
    ],

    "Item": [
        "Milk", "Bread", "Diaper",
        "Bread", "Coke", "Diaper",
        "Milk", "Coke",
        "Milk", "Bread", "Coke"
    ]
}

df = pd.DataFrame(data)

print("Original Data:")
print(df)


# ============================================
# 2. CREATE TID SETS
# ============================================

tid_sets = {}

for item in df["Item"].unique():

    transactions = set(
        df[df["Item"] == item]["TransactionID"]
    )

    tid_sets[item] = transactions


print("\nTID Sets:")

for item, tids in tid_sets.items():
    print(item, ":", tids)


# ============================================
# 3. SET MINIMUM SUPPORT
# ============================================

min_support = 0.5

total_transactions = df["TransactionID"].nunique()

min_count = min_support * total_transactions

print("\nTotal Transactions:", total_transactions)
print("Minimum Support:", min_support)
print("Minimum Count:", min_count)


# ============================================
# 4. FIND FREQUENT 1-ITEMSETS
# ============================================

frequent_itemsets = {}

for item, tids in tid_sets.items():

    support = len(tids) / total_transactions

    if support >= min_support:

        frequent_itemsets[frozenset([item])] = {
            "tids": tids,
            "support": support
        }


print("\nFrequent 1-Itemsets:")

for itemset, value in frequent_itemsets.items():

    print(
        set(itemset),
        "Support:",
        value["support"]
    )


# ============================================
# 5. ECLAT FUNCTION
# ============================================

def eclat(itemsets, min_support, total_transactions):

    all_frequent = itemsets.copy()

    current_itemsets = itemsets

    while current_itemsets:

        next_itemsets = {}

        keys = list(current_itemsets.keys())

        # Generate combinations
        for i in range(len(keys)):

            for j in range(i + 1, len(keys)):

                itemset1 = keys[i]
                itemset2 = keys[j]

                # Combine itemsets
                new_itemset = itemset1 | itemset2

                # Only create itemsets larger than current ones
                if len(new_itemset) != len(itemset1) + 1:
                    continue

                # Intersect TID sets
                tids1 = current_itemsets[itemset1]["tids"]
                tids2 = current_itemsets[itemset2]["tids"]

                new_tids = tids1 & tids2

                # Calculate support
                support = len(new_tids) / total_transactions

                # Check minimum support
                if support >= min_support:

                    if new_itemset not in next_itemsets:

                        next_itemsets[new_itemset] = {
                            "tids": new_tids,
                            "support": support
                        }

        # Add new frequent itemsets
        all_frequent.update(next_itemsets)

        current_itemsets = next_itemsets

    return all_frequent


# ============================================
# 6. RUN ECLAT
# ============================================

frequent_itemsets = eclat(
    frequent_itemsets,
    min_support,
    total_transactions
)


# ============================================
# 7. DISPLAY FREQUENT ITEMSETS
# ============================================

print("\n============================================")
print("FREQUENT ITEMSETS")
print("============================================")

for itemset, value in frequent_itemsets.items():

    print(
        set(itemset),
        "→ Support:",
        round(value["support"], 2),
        "→ TIDs:",
        value["tids"]
    )

ECLAT : ------------------->
Original Data:
    TransactionID    Item
0               1    Milk
1               1   Bread
2               1  Diaper
3               2   Bread
4               2    Coke
5               2  Diaper
6               3    Milk
7               3    Coke
8               4    Milk
9               4   Bread
10              4    Coke

TID Sets:
Milk : {1, 3, 4}
Bread : {1, 2, 4}
Diaper : {1, 2}
Coke : {2, 3, 4}

Total Transactions: 4
Minimum Support: 0.5
Minimum Count: 2.0

Frequent 1-Itemsets:
{'Milk'} Support: 0.75
{'Bread'} Support: 0.75
{'Diaper'} Support: 0.5
{'Coke'} Support: 0.75

FREQUENT ITEMSETS
{'Milk'} → Support: 0.75 → TIDs: {1, 3, 4}
{'Bread'} → Support: 0.75 → TIDs: {1, 2, 4}
{'Diaper'} → Support: 0.5 → TIDs: {1, 2}
{'Coke'} → Support: 0.75 → TIDs: {2, 3, 4}
{'Milk', 'Bread'} → Support: 0.5 → TIDs: {1, 4}
{'Milk', 'Coke'} → Support: 0.5 → TIDs: {3, 4}
{'Diaper', 'Bread'} → Support: 0.5 → TIDs: {1, 2}
{'Coke', 'Bread'} → Support: 0.5 → TIDs: {2, 4}
